# DS2002 · Pandas Challenge

**Lab — 2026-09-18 · Fall 2026**  
**Ciara Graves**

---

## Lab 04 — Pandas Challenge

Four hundred generated orders. Each question builds toward a demand report you could hand a vendor.

The data is seeded, so everyone's numbers should match. That is deliberate: if your total revenue differs from your neighbor's, one of you has a bug, and the assertions at the end will tell you which.

Every answer needs the number **and** a sentence saying what it means. A cell that prints `4218.5` with no interpretation is half an answer.

In [1]:
import pandas as pd, numpy as np
rng = np.random.default_rng(4)
n = 400
df = pd.DataFrame({
    'vendor_id': rng.choice(['V-01','V-05','V-10','V-18'], n),
    'category': rng.choice(['Food','Merch','RainGear','Drink'], n, p=[.5,.2,.1,.2]),
    'qty': rng.integers(1, 4, n),
    'price': rng.choice([4.5, 6.0, 7.5, 12.0, 24.0], n),
})
df.head()

,vendor_id,category,qty,price
0,V-10,Drink,2,24.0
1,V-18,RainGear,1,12.0
2,V-18,Drink,3,4.5
3,V-10,Food,2,12.0
4,V-18,Drink,3,7.5


### Q1 — Add `revenue`, then report total revenue and total units.

*Expected: 400 rows, and revenue should land between $8,000 and $9,000.*

In [2]:
# TODO
# add revenue to df
df['revenue'] = df['qty'] * df['price']
#df.head()

# report total revenue and total units
total_revenue = df['revenue'].sum()
total_units = df['qty'].sum()

print('Total revenue:', total_revenue)
print('Total units:', total_units)
print('Rows:', len(df)) # check for the expected number of rows


Total revenue: 8520.0
Total units: 783
Rows: 400


**Q1 Explanation:** The output of `$8,520` for total revenue means that in our dataframe, the total revenue across all vendors is `$8,520`. The value for total untis being 783 means that across the 400 rows of the dataframe, the total number of units sold is 783 units. Both values help quantify and describe information our dataframe has.

### Q2 — Revenue by category, highest to lowest.

Include the share of total as a percentage in the same table.

In [3]:
# TODO
# want to use group by
by_category = (
    df.groupby('category')['revenue']
      .sum()
      .reset_index())

by_category['share_of_total'] = (
    by_category['revenue'] / df['revenue'].sum() * 100)

by_category = by_category.sort_values(
    'revenue', ascending=False)
# output
by_category

,category,revenue,share_of_total
1,Food,4293.0,50.387324
2,Merch,1771.5,20.792254
0,Drink,1554.0,18.239437
3,RainGear,901.5,10.580986


**Q3 Explanation:** The food category earns the most money, with 50.89% of the total revenue. Rain gear earns the least, which makes sense because consumers will only purchase rain gear when poor weather conditions arise, whereas food, merch, and drink sales are more consistent.

### Q3 — Which vendor has the highest *average* order revenue?

Report the average alongside the order count for each vendor. A high average on twelve orders is a different claim from a high average on two hundred.

In [4]:
# TODO
vendor_summary = (
    df.groupby('vendor_id')
      .agg(
          avg_order_revenue=('revenue', 'mean'),
          order_count=('revenue', 'count'))
      .sort_values('avg_order_revenue', ascending=False))
# output
vendor_summary

,avg_order_revenue,order_count
vendor_id,,
V-01,22.595745,94
V-18,21.750000,108
V-05,20.580645,93
V-10,20.314286,105


**Q3 Explanation:** The vendor with id VD-01 has the highest average order revenue with an order count of 94. The other 4 highest vendors have order counts ranging from 93-108. For VD-01, this suggests that it's not "luxury" vendor that few customers would by from, but rather a reasonably popular vendor with a good revenue stream.

### Q4 — What share of revenue comes from Merch?

Print it as a percentage rounded to one decimal.

In [5]:
# TODO
merch_revenue = df.loc[df['category'] == 'Merch', 'revenue'].sum()
# find share from total rev
merch_share = merch_revenue / df['revenue'].sum() * 100
merch_share = round(merch_share, 1)
print(f'Share of revenue for merch: {merch_share}%')

Share of revenue for merch: 20.8%


**Q4 Explanation:** Merch accounts for 20.8% of the total revenue. It would be a good goal to see this percentage increase, merchandising has a large potential for high profit & revenue.

### Q5 — Join in the vendor names.

The frame only has `vendor_id`. Merge the lookup below so your report is readable.

**Requirements:** left join, `validate='many_to_one'`, and prove the row count and revenue total did not change. One vendor id in the orders is not in this lookup — find it, and decide what to do about it.

In [6]:
vendor_names = pd.DataFrame({
    'vendor_id': ['V-01', 'V-05', 'V-10'],
    'vendor_name': ['Hoos Burgers', 'Rotunda Tacos', 'Cav Merch North'],
})

# TODO: merge, validate, and report the unmatched vendor
# save totals prior to merge, will check later
rows_before = len(df)
revenue_before = df['revenue'].sum()
#left join vendor names
joined = df.merge(
    vendor_names,
    on='vendor_id',
    how='left',
    validate='many_to_one')
# print rows before and after to validate
print('Rows before:', rows_before)
print('Rows after:', len(joined))
print('Revenue before:', revenue_before)
print('Revenue after:', joined['revenue'].sum())
# find the unmatched vendor
unmatched = joined.loc[joined['vendor_name'].isna(), 'vendor_id'].unique()
print('Unmatched vendor:', unmatched)
# give unknown vendor the name Unknown
joined.loc[joined['vendor_id'] == 'V-18', 'vendor_name'] = 'Unknown'


Rows before: 400
Rows after: 400
Revenue before: 8520.0
Revenue after: 8520.0
Unmatched vendor: ['V-18']


**The unmatched vendor, and what I did about it:** The unmatched vendor was V-18, and I chose to assign it the name "Unknown".

### Q6 — A pivot table: vendors down the side, categories across the top, revenue in the cells.

Add row and column totals so it reads as a report rather than a grid of numbers.

In [7]:
# TODO
df_pivot = pd.pivot_table(
    joined,
    values='revenue',
    index='vendor_name',
    columns='category',
    aggfunc='sum',
    fill_value=0,
    margins=True,
    margins_name='Total')
# output
df_pivot

category,Drink,Food,Merch,RainGear,Total
vendor_name,,,,,
Cav Merch North,502.5,1054.5,400.5,175.5,2133.0
Hoos Burgers,171.0,1338.0,373.5,241.5,2124.0
Rotunda Tacos,298.5,882.0,489.0,244.5,1914.0
Unknown,582.0,1018.5,508.5,240.0,2349.0
Total,1554.0,4293.0,1771.5,901.5,8520.0


**Q6 Explanation:** From the table we can notice that Hoos Burgers, the second most profitable vendor, sells much less in the drink category than the topo vendor, Cav Merch North. This indicates that Hoos Burgers should better advertise and incentivize their drinks.

### Q7 — Validate your work

**TODO:** uncomment and make these pass. Assign your results to the named variables as you go.

In [8]:
assert len(df) == 400
assert 8000 < df['revenue'].sum() < 9000, df['revenue'].sum()
assert abs(by_category['revenue'].sum() - df['revenue'].sum()) < 0.01
assert len(joined) == len(df), 'the vendor merge changed the row count'
print('checks passed.')

checks passed.


**Q7 Explanation:** All checks passed, work/outputs are verified.

### Write-up

**a)** What would you tell these vendors to do differently next game? One paragraph, with at least two numbers from your report in it.

**b)** Which of your seven answers is the least trustworthy, and why? Point at a specific weakness — a small group size, an unmatched vendor, a category that is really two things.

a) I would suggest directly to Hoos Burgers that they should better market this drinks. Whether this means carrying a larger selection of beverages or having a drink-deal to better promote their beverages. Their total revenue is only 9 dollars less than Cav Merch North, the top vendor, yet their revenue from drink sales is 331.50 dollars less; suggesting that they are loosing out on sales. Further, merch sales only make up 20.8% of the total revenue, so I would suggest that all vendors try to improve upon this category. Merchandise is a great way to improve revenue, if it is properly marketed, customers will flock at the idea to by a piece of merch to remember their experience by.

b) Of the seven questions, I would say Q3: finding the vendor with the highest order revenue has the least value. While seeking out and ordering vendors on their average order revenue can provide some insight to why some vendors may be more successful than others, the statisitcs lack depth because they don't tell the full story. A vendor that sells items with very high profit margins but is infrequently bought could appear to be more successful than they actually are. It is hard to direct the reader to a clear relationship when answering that question because the statistic lacks depth. Thus, I would consider that answer to be least trustworthy.